# B (CSV) — QLTT: Quantile Risk from Loss CSV
Each row is a hyperparameter; columns `loss_1..loss_n` are calibration errors (real-valued OK).

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv
from utils.testing import quantile_exceedances_pvalue, holm_bonferroni
csv_path = 'data/sample_real_losses.csv'  # <- change to your file
tau = 0.9                                 # target quantile (e.g., 90th pct)
q_star = 1.5                               # threshold for that quantile
alpha_mtp = 0.05                           # FWER level


In [ ]:
ids, L, cols = load_losses_csv(csv_path)
m, n = L.shape
rows = []
for i in range(m):
    losses = L[i]
    p = quantile_exceedances_pvalue(losses, target_q=q_star, tau=tau)
    rows.append({'hyperparam_id': ids[i], 'emp_tau_quantile': float(np.quantile(losses, tau)), 'pval': float(p)})
df = pd.DataFrame(rows)
df['selected'] = holm_bonferroni(df['pval'].values, alpha=alpha_mtp)
df.sort_values(['selected','emp_tau_quantile'], ascending=[False, True])

**How to use with your data**
- Put your CSV in `data/`.
- Ensure first column is `hyperparam_id` and losses are `loss_1...loss_n`.
- Set `tau` and `q_star` per your SLA.
